# 🕐 Time Series Feature Engineering

> **Folder:** `09_Time_Series_Machine_Learning`  
> **Notebook:** `feature_engineering.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Build **lag features** correctly without lookahead bias
- Compute **rolling window statistics** (mean, std, min, max, skew)
- Create **expanding window** (cumulative) features
- Engineer **calendar features** with proper **cyclical encoding**
- Apply **stationarity transforms** (differencing, log, Box-Cox)
- Detect seasonality and build **seasonal features**
- Perform **autocorrelation analysis** (ACF / PACF)
- Build a complete **feature matrix** ready for ML models
- Use **TimeSeriesSplit** correctly to avoid leakage

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Synthetic Dataset | Trend + seasonality + noise |
| 2 | EDA & Stationarity Tests | ADF, KPSS, rolling stats |
| 3 | Differencing & Log Transform | Making series stationary |
| 4 | Decomposition | Trend / Seasonal / Residual |
| 5 | Lag Features | Respecting forecast horizon |
| 6 | Rolling Window Features | Mean, std, min, max, skew |
| 7 | Expanding Window Features | Cumulative statistics |
| 8 | Calendar Features | Cyclical encoding |
| 9 | ACF / PACF Analysis | Lag selection guidance |
| 10 | Full Feature Matrix | Complete ML-ready dataset |
| 11 | TimeSeriesSplit CV | Leakage-free evaluation |
| 12 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats

try:
    from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
    from statsmodels.tsa.seasonal import seasonal_decompose, STL
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    STATSMODELS = True
except ImportError:
    STATSMODELS = False
    print("statsmodels not installed — pip install statsmodels")

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded!')

---
## 1️⃣ Synthetic Time Series Dataset

> We build a realistic synthetic daily series with:
> - **Linear trend** (slowly rising mean)
> - **Weekly seasonality** (weekday pattern)
> - **Yearly seasonality** (annual cycle)
> - **Gaussian noise** (random fluctuations)
> - **A few outliers** (real-world contamination)

> This mirrors retail sales, energy demand, web traffic, or financial data.


In [ ]:
np.random.seed(42)

# ── Build synthetic daily time series (3 years) ───────────────────────────
n_days  = 365 * 3
dates   = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
t       = np.arange(n_days)

# Components
trend      = 100 + 0.05 * t
weekly_s   = 15 * np.sin(2 * np.pi * t / 7)          # weekly cycle
yearly_s   = 30 * np.sin(2 * np.pi * t / 365.25)     # yearly cycle
noise      = np.random.normal(0, 8, n_days)

# Combine
value = trend + weekly_s + yearly_s + noise

# Add outliers
outlier_idx = np.random.choice(n_days, size=15, replace=False)
value[outlier_idx] += np.random.choice([-60, 60], size=15)

# Build DataFrame
df = pd.DataFrame({'value': value}, index=dates)
df.index.name = 'date'
df.index.freq = 'D'

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df.index[0].date()} to {df.index[-1].date()}')
print(f'Value stats   :')
print(df['value'].describe().round(2).to_string())

# Plot
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=False)

axes[0].plot(df.index, df['value'], color=COLORS['primary'],
             linewidth=0.9, alpha=0.85)
axes[0].set_title('Full Time Series — 3 Years Daily Data',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=10)

axes[1].plot(df.index[:90], df['value'].iloc[:90],
             color=COLORS['accent'], linewidth=1.5)
axes[1].set_title('First 90 Days — Weekly Pattern Visible',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=10)

axes[2].plot(df.index[:365], df['value'].iloc[:365],
             color=COLORS['secondary'], linewidth=1.2)
axes[2].set_title('First Year — Annual Seasonality Visible',
                  fontsize=12, fontweight='bold')
axes[2].set_ylabel('Value', fontsize=10)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Synthetic Time Series — Trend + Seasonality + Noise',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2️⃣ EDA & Stationarity Tests

> Before any feature engineering, check whether the series is **stationary**.
>
> **ADF Test:** H₀ = non-stationary. p < 0.05 → stationary ✅  
> **KPSS Test:** H₀ = stationary. p < 0.05 → non-stationary ❌
>
> Use both: if ADF says stationary AND KPSS says stationary → confirmed.


In [ ]:
# Rolling statistics
roll_mean = df['value'].rolling(30).mean()
roll_std  = df['value'].rolling(30).std()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(df.index, df['value'], color=COLORS['primary'],
             linewidth=0.8, alpha=0.7, label='Original')
axes[0].plot(df.index, roll_mean, color=COLORS['secondary'],
             linewidth=2.5, label='30-day rolling mean')
axes[0].fill_between(df.index,
                     roll_mean - roll_std,
                     roll_mean + roll_std,
                     alpha=0.15, color=COLORS['secondary'],
                     label='±1 rolling std')
axes[0].set_title('Rolling Mean and Std — Non-stationary trend visible',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(df.index, roll_std, color=COLORS['warning'],
             linewidth=1.5, label='30-day rolling std')
axes[1].axhline(roll_std.mean(), color='black', linestyle='--',
                linewidth=1.5, label=f'Mean std={roll_std.mean():.2f}')
axes[1].set_title('Rolling Std — Variance roughly stable (no heteroskedasticity)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('Stationarity EDA', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Statistical stationarity tests ────────────────────────────────────────
if STATSMODELS:
    print('Stationarity Tests — Original Series')
    print('=' * 50)

    # ADF Test
    adf_result = adfuller(df['value'].dropna(), autolag='AIC')
    print(f'ADF Statistic : {adf_result[0]:.4f}')
    print(f'ADF p-value   : {adf_result[1]:.4f}')
    print(f'ADF Result    : {"STATIONARY ✅" if adf_result[1] < 0.05 else "NON-STATIONARY ❌"}')
    print()

    # KPSS Test
    try:
        kpss_result = kpss(df['value'].dropna(), regression='c', nlags='auto')
        print(f'KPSS Statistic: {kpss_result[0]:.4f}')
        print(f'KPSS p-value  : {kpss_result[1]:.4f}')
        print(f'KPSS Result   : {"NON-STATIONARY ❌" if kpss_result[1] < 0.05 else "STATIONARY ✅"}')
    except Exception as e:
        print(f'KPSS: {e}')
else:
    print('statsmodels not available — skipping formal stationarity tests')
    print('Visual inspection: rolling mean has upward trend -> non-stationary')

---
## 3️⃣ Differencing & Log Transform — Making Series Stationary

> **Differencing** removes trend by computing changes between consecutive values.  
> **Log transform** stabilizes variance when it grows with the level.
>
> After differencing, re-run stationarity tests to confirm.


In [ ]:
# First-order differencing
df['diff1']    = df['value'].diff(1)

# Seasonal differencing (period=7 for weekly)
df['diff7']    = df['value'].diff(7)

# Combined: seasonal + first-order
df['diff7_1']  = df['value'].diff(7).diff(1)

# Log transform (value > 0 here — shift if needed)
df['log_val']  = np.log(df['value'] - df['value'].min() + 1)
df['log_diff'] = df['log_val'].diff(1)

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
series_configs = [
    ('value',   'Original Series',        COLORS['primary']),
    ('diff1',   'First-Order Diff (d=1)', COLORS['secondary']),
    ('diff7',   'Seasonal Diff (d=7)',    COLORS['accent']),
    ('diff7_1', 'Seasonal + First Diff',  COLORS['warning']),
]
for ax, (col, title, color) in zip(axes, series_configs):
    ax.plot(df.index, df[col], color=color, linewidth=0.9, alpha=0.85)
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Value', fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('Differencing Strategies — Achieving Stationarity',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ADF on differenced series
if STATSMODELS:
    print('ADF Tests After Differencing:')
    for col, label in [('diff1','First-diff'), ('diff7','Seasonal-diff(7)'),
                       ('diff7_1','Seasonal+First')]:
        series_clean = df[col].dropna()
        adf = adfuller(series_clean, autolag='AIC')
        result = 'STATIONARY ✅' if adf[1] < 0.05 else 'NON-STATIONARY ❌'
        print(f'  {label:25s}: p={adf[1]:.4f}  {result}')

---
## 4️⃣ Seasonal Decomposition — Trend / Seasonal / Residual

> Decomposition separates the series into interpretable components.  
> Use the **residual** as the cleaned signal for feature engineering  
> when you want to remove trend and seasonality before ML.


In [ ]:
if STATSMODELS:
    # Classical decomposition
    decomp = seasonal_decompose(df['value'], model='additive', period=7)

    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    components = [
        (df['value'],      'Original',  COLORS['primary']),
        (decomp.trend,     'Trend',     COLORS['secondary']),
        (decomp.seasonal,  'Seasonal',  COLORS['accent']),
        (decomp.resid,     'Residual',  COLORS['warning']),
    ]
    for ax, (series, title, color) in zip(axes, components):
        ax.plot(df.index, series, color=color, linewidth=0.9)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel('Value', fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    plt.suptitle('Classical Seasonal Decomposition (additive, period=7)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    # Store components for feature use
    df['trend_comp']    = decomp.trend
    df['seasonal_comp'] = decomp.seasonal
    df['residual_comp'] = decomp.resid

    print(f'Trend range   : [{decomp.trend.dropna().min():.2f}, {decomp.trend.dropna().max():.2f}]')
    print(f'Seasonal range: [{decomp.seasonal.min():.2f}, {decomp.seasonal.max():.2f}]')
    print(f'Residual std  : {decomp.resid.dropna().std():.4f}')
else:
    print('statsmodels not available — using manual trend/seasonal extraction')
    df['trend_comp']    = df['value'].rolling(30, center=True).mean()
    df['seasonal_comp'] = df['value'] - df['trend_comp']
    df['residual_comp'] = df['value'] - df['trend_comp'] - df['seasonal_comp']

---
## 5️⃣ Lag Features — The Most Important Time Series Feature

> Lag features capture **past values** of the target as predictors.
>
> **Golden Rule:** Only use lags ≥ forecast horizon (h).
> - Forecasting 1-step ahead (h=1): use lag_1, lag_2, lag_7, ...
> - Forecasting 7-step ahead (h=7): use lag_7, lag_8, lag_14, ... NOT lag_1
>
> Using a lag shorter than the forecast horizon is **lookahead bias**!


In [ ]:
# ── Forecast horizon = 1 (next-day prediction) ───────────────────────────
h = 1  # forecast horizon

# Lags relative to horizon
lag_values = [1, 2, 3, 7, 14, 21, 28, 35, 42]
for lag in lag_values:
    if lag >= h:
        df[f'lag_{lag}'] = df['value'].shift(lag)

print(f'Lag features created (horizon={h}):')
lag_cols = [c for c in df.columns if c.startswith('lag_')]
print(f'  {lag_cols}')

# Lag correlation with target
lag_corr = {}
for col in lag_cols:
    mask = df[col].notna() & df['value'].notna()
    corr = df.loc[mask, 'value'].corr(df.loc[mask, col])
    lag_corr[col] = corr

corr_df = pd.DataFrame(list(lag_corr.items()),
                        columns=['Lag Feature', 'Correlation with Target']
                       ).sort_values('Correlation with Target', key=abs, ascending=False)

print('
Lag-Target Correlation:')
print(corr_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))

bar_colors = [COLORS['accent'] if v > 0 else COLORS['secondary']
              for v in corr_df['Correlation with Target']]
axes[0].barh(corr_df['Lag Feature'], corr_df['Correlation with Target'],
             color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_xlabel('Pearson Correlation', fontsize=11)
axes[0].set_title('Lag Features — Correlation with Target',
                  fontsize=12, fontweight='bold')

# Scatter: lag_1 vs target
mask = df['lag_1'].notna()
axes[1].scatter(df.loc[mask, 'lag_1'], df.loc[mask, 'value'],
                alpha=0.3, s=10, color=COLORS['primary'])
axes[1].set_xlabel('lag_1 (yesterday)', fontsize=11)
axes[1].set_ylabel('value (today)', fontsize=11)
axes[1].set_title(f'lag_1 vs Target (corr={lag_corr["lag_1"]:.3f})',
                  fontsize=12, fontweight='bold')

plt.suptitle('Lag Features', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6️⃣ Rolling Window Features — Local Statistics

> Rolling windows capture **local temporal patterns**:
> - Mean → smoothed level
> - Std → local volatility
> - Min/Max → recent extremes
> - Skewness → distribution shape changes
>
> ⚠️ Always use `shift(h)` before rolling to avoid lookahead:
> `df['value'].shift(h).rolling(w).mean()`


In [ ]:
h = 1  # forecast horizon — shift before rolling!

windows = [7, 14, 28, 90]

for w in windows:
    base = df['value'].shift(h)   # shift first to avoid lookahead
    df[f'roll_mean_{w}']  = base.rolling(w).mean()
    df[f'roll_std_{w}']   = base.rolling(w).std()
    df[f'roll_min_{w}']   = base.rolling(w).min()
    df[f'roll_max_{w}']   = base.rolling(w).max()
    df[f'roll_range_{w}'] = df[f'roll_max_{w}'] - df[f'roll_min_{w}']
    df[f'roll_skew_{w}']  = base.rolling(w).skew()
    df[f'roll_kurt_{w}']  = base.rolling(w).kurt()

roll_cols = [c for c in df.columns if c.startswith('roll_')]
print(f'Rolling features created: {len(roll_cols)}')
print(f'  {roll_cols[:10]} ...')

# Visualize rolling means
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(df.index, df['value'], color=COLORS['primary'],
             linewidth=0.8, alpha=0.6, label='Original')
for w, color in zip(windows, COLORS['palette'][:4]):
    axes[0].plot(df.index, df[f'roll_mean_{w}'],
                 linewidth=2, color=color, label=f'{w}-day MA')
axes[0].set_title('Rolling Mean — Multiple Windows',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylabel('Value', fontsize=10)

for w, color in zip(windows, COLORS['palette'][:4]):
    axes[1].plot(df.index, df[f'roll_std_{w}'],
                 linewidth=1.5, color=color, label=f'{w}-day Std')
axes[1].set_title('Rolling Std — Local Volatility',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].set_ylabel('Std', fontsize=10)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('Rolling Window Features', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7️⃣ Expanding Window Features — Cumulative Statistics

> Expanding windows use **all past data** up to the current point —  
> useful for tracking long-run averages and cumulative quantities.
>
> ⚠️ Same rule: `shift(h)` before `.expanding()` to avoid lookahead.


In [ ]:
h = 1
base_exp = df['value'].shift(h)

df['exp_mean']  = base_exp.expanding().mean()
df['exp_std']   = base_exp.expanding().std()
df['exp_min']   = base_exp.expanding().min()
df['exp_max']   = base_exp.expanding().max()
df['exp_range'] = df['exp_max'] - df['exp_min']

# Ratio: current vs historical mean
df['val_to_exp_mean'] = df['value'].shift(h) / df['exp_mean']

# Percent rank (quantile of current value in historical distribution)
df['exp_pct_rank'] = base_exp.expanding().rank(pct=True)

print('Expanding window features:')
exp_cols = [c for c in df.columns if c.startswith('exp_') or
            c == 'val_to_exp_mean' or c == 'exp_pct_rank']
print(f'  {exp_cols}')

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
axes[0].plot(df.index, df['value'], color=COLORS['primary'],
             linewidth=0.8, alpha=0.6, label='Original')
axes[0].plot(df.index, df['exp_mean'], color=COLORS['secondary'],
             linewidth=2.5, label='Expanding mean')
axes[0].fill_between(df.index,
                     df['exp_mean'] - df['exp_std'],
                     df['exp_mean'] + df['exp_std'],
                     alpha=0.12, color=COLORS['secondary'])
axes[0].set_title('Expanding Mean (cumulative historical average)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(df.index, df['exp_pct_rank'], color=COLORS['accent'],
             linewidth=1.2, alpha=0.8)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1.5,
                label='Median (50th pct)')
axes[1].set_title('Expanding Percentile Rank — Where is today vs history?',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].set_ylim([0, 1])

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.suptitle('Expanding Window Features', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8️⃣ Calendar Features — Cyclical Encoding

> Calendar features capture **temporal patterns** tied to the calendar:
> - Day of week (weekday effect)
> - Month of year (seasonal effect)
> - Day of year (annual cycle)
>
> **Cyclical encoding:** encode periodic features as sin/cos pairs so  
> December (12) is treated as close to January (1):
>
> `sin_month = sin(2π × month / 12)`  
> `cos_month = cos(2π × month / 12)`
>
> ⚠️ Never use raw integer month (1–12) as a feature — the model thinks  
> December and January are far apart!


In [ ]:
# ── Raw calendar features ─────────────────────────────────────────────────
df['dayofweek']  = df.index.dayofweek           # 0=Mon, 6=Sun
df['month']      = df.index.month               # 1–12
df['quarter']    = df.index.quarter             # 1–4
df['dayofyear']  = df.index.dayofyear           # 1–365
df['weekofyear'] = df.index.isocalendar().week.astype(int)
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
df['is_month_start'] = df.index.is_month_start.astype(int)
df['is_month_end']   = df.index.is_month_end.astype(int)

# ── Cyclical encoding ──────────────────────────────────────────────────────
# Day of week (period=7)
df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

# Month (period=12)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Day of year (period=365)
df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)

print('Calendar features:')
cal_cols = ['dayofweek','month','quarter','dayofyear','is_weekend',
            'dow_sin','dow_cos','month_sin','month_cos','doy_sin','doy_cos']
print(f'  {cal_cols}')

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# Mean value by day of week
dow_mean = df.groupby('dayofweek')['value'].mean()
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
axes[0,0].bar(dow_labels, dow_mean.values, color=COLORS['primary'], alpha=0.85)
axes[0,0].set_title('Mean Value by Day of Week', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Mean Value')

# Mean value by month
month_mean = df.groupby('month')['value'].mean()
axes[0,1].bar(range(1,13), month_mean.values, color=COLORS['accent'], alpha=0.85)
axes[0,1].set_xlabel('Month'); axes[0,1].set_title('Mean Value by Month',
                                                     fontsize=11, fontweight='bold')

# Cyclical encoding — day of week
theta = 2 * np.pi * np.arange(7) / 7
axes[0,2].scatter(np.sin(theta), np.cos(theta),
                  s=200, c=COLORS['palette'][:7], zorder=5)
for i, label in enumerate(dow_labels):
    axes[0,2].annotate(label, (np.sin(theta[i]), np.cos(theta[i])),
                       textcoords='offset points', xytext=(8,4), fontsize=10)
circle = plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--')
axes[0,2].add_patch(circle)
axes[0,2].set_xlim([-1.5,1.5]); axes[0,2].set_ylim([-1.5,1.5])
axes[0,2].set_aspect('equal')
axes[0,2].set_title('Cyclical Encoding — Day of Week
(Mon and Sun are adjacent!)',
                    fontsize=11, fontweight='bold')

# Cyclical encoding — month
theta_m = 2 * np.pi * np.arange(1,13) / 12
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
sc = axes[1,0].scatter(np.sin(theta_m), np.cos(theta_m),
                        s=150, c=range(12), cmap='hsv', zorder=5)
for i, mn in enumerate(month_names):
    axes[1,0].annotate(mn, (np.sin(theta_m[i]), np.cos(theta_m[i])),
                       textcoords='offset points', xytext=(5,3), fontsize=9)
circle2 = plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--')
axes[1,0].add_patch(circle2)
axes[1,0].set_xlim([-1.5,1.5]); axes[1,0].set_ylim([-1.5,1.5])
axes[1,0].set_aspect('equal')
axes[1,0].set_title('Cyclical Encoding — Month
(Dec and Jan are adjacent!)',
                    fontsize=11, fontweight='bold')

# sin vs cos scatter (monthly)
sc2 = axes[1,1].scatter(df['month_sin'], df['month_cos'],
                         c=df['month'], cmap='hsv', s=5, alpha=0.4)
plt.colorbar(sc2, ax=axes[1,1], label='Month')
axes[1,1].set_xlabel('month_sin'); axes[1,1].set_ylabel('month_cos')
axes[1,1].set_title('Monthly Cyclical Features in 2D Space',
                    fontsize=11, fontweight='bold')

# Day of year sin encoding
axes[1,2].plot(range(1,366),
               [np.sin(2*np.pi*d/365.25) for d in range(1,366)],
               color=COLORS['primary'], linewidth=2, label='doy_sin')
axes[1,2].plot(range(1,366),
               [np.cos(2*np.pi*d/365.25) for d in range(1,366)],
               color=COLORS['secondary'], linewidth=2, label='doy_cos')
axes[1,2].set_xlabel('Day of Year')
axes[1,2].set_title('Day-of-Year Cyclical Features', fontsize=11, fontweight='bold')
axes[1,2].legend(fontsize=9)

plt.suptitle('Calendar Features — Cyclical Encoding',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9️⃣ ACF & PACF — Lag Selection Guidance

> **ACF (Autocorrelation Function):** correlation of series with its own lags.  
> → Spikes at lag k → MA(q) component in ARIMA
>
> **PACF (Partial Autocorrelation Function):** correlation after removing  
> effects of shorter lags.  
> → Spikes at lag k → AR(p) component in ARIMA
>
> **For ML feature selection:** large ACF / PACF spikes → include that lag as feature.


In [ ]:
if STATSMODELS:
    fig, axes = plt.subplots(2, 2, figsize=(17, 9))

    # Original series
    plot_acf(df['value'].dropna(), lags=60, ax=axes[0,0],
             color=COLORS['primary'], alpha=0.05)
    axes[0,0].set_title('ACF — Original Series', fontsize=12, fontweight='bold')

    plot_pacf(df['value'].dropna(), lags=60, ax=axes[0,1],
              color=COLORS['primary'], alpha=0.05, method='ywm')
    axes[0,1].set_title('PACF — Original Series', fontsize=12, fontweight='bold')

    # Differenced series
    diff1 = df['value'].diff(1).dropna()
    plot_acf(diff1, lags=60, ax=axes[1,0],
             color=COLORS['accent'], alpha=0.05)
    axes[1,0].set_title('ACF — First-Differenced Series',
                        fontsize=12, fontweight='bold')

    plot_pacf(diff1, lags=60, ax=axes[1,1],
              color=COLORS['accent'], alpha=0.05, method='ywm')
    axes[1,1].set_title('PACF — First-Differenced Series',
                        fontsize=12, fontweight='bold')

    plt.suptitle('ACF & PACF — Lag Structure Analysis',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    # Compute ACF values for feature importance insight
    acf_vals  = acf(df['value'].dropna(), nlags=42)
    pacf_vals = pacf(df['value'].dropna(), nlags=42, method='ywm')

    print('Top ACF lags (absolute value):')
    acf_df = pd.DataFrame({'Lag': range(1, 43),
                           'ACF': acf_vals[1:]}).sort_values('ACF', key=abs, ascending=False)
    print(acf_df.head(10).to_string(index=False))
    print('
Key lags to include as features:',
          acf_df.head(5)['Lag'].tolist())
else:
    print('statsmodels not available — using correlation-based lag analysis')
    lag_corr_manual = {lag: df['value'].corr(df['value'].shift(lag))
                       for lag in range(1, 43)}
    top_lags = sorted(lag_corr_manual, key=lambda x: abs(lag_corr_manual[x]),
                      reverse=True)[:5]
    print(f'Top correlated lags: {top_lags}')

---
## 🔟 Full Feature Matrix — ML-Ready Dataset

> Combine all engineered features into a single ML-ready DataFrame.  
> Drop rows with NaN (created by lags/rolling windows) and split temporally.


In [ ]:
# ── Assemble feature matrix ───────────────────────────────────────────────
h       = 1   # forecast horizon
df['target'] = df['value'].shift(-h)  # predict h steps ahead

feature_cols = (
    # Lag features
    [f'lag_{l}' for l in [1,2,3,7,14,21,28]] +
    # Rolling stats
    [f'roll_mean_{w}' for w in [7,14,28,90]] +
    [f'roll_std_{w}'  for w in [7,14,28]] +
    [f'roll_min_{w}'  for w in [7,28]] +
    [f'roll_max_{w}'  for w in [7,28]] +
    # Expanding
    ['exp_mean','exp_std','exp_pct_rank','val_to_exp_mean'] +
    # Calendar (cyclical)
    ['dow_sin','dow_cos','month_sin','month_cos','doy_sin','doy_cos'] +
    # Calendar (categorical)
    ['is_weekend','is_month_end','quarter']
)

# Only keep cols that exist in df
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df['target'].copy()

# Drop rows with NaN (from lags/rolling)
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]; y = y[mask]

print(f'Feature matrix shape : {X.shape}')
print(f'Target shape         : {y.shape}')
print(f'Date range           : {X.index[0].date()} to {X.index[-1].date()}')
print(f'Features             : {len(feature_cols)}')
print(f'\nFeature sample (first 3 rows):')
print(X.head(3).round(3).to_string())

# Correlation heatmap (top features)
top_corr = X.corrwith(y).abs().sort_values(ascending=False).head(15)
print(f'\nTop 15 features by correlation with target:')
print(top_corr.round(4).to_string())

fig, ax = plt.subplots(figsize=(13, 5))
bar_c = [COLORS['accent'] if 'lag' in c else
         COLORS['primary'] if 'roll' in c else
         COLORS['warning'] for c in top_corr.index]
ax.barh(top_corr.index, top_corr.values, color=bar_c, alpha=0.85, edgecolor='white')
ax.set_xlabel('|Pearson Correlation with Target|', fontsize=11)
ax.set_title('Top 15 Features by Correlation with Target',
             fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['accent'],  label='Lag features'),
    Patch(color=COLORS['primary'], label='Rolling features'),
    Patch(color=COLORS['warning'], label='Calendar features'),
], fontsize=9)
plt.tight_layout(); plt.show()

---
## 1️⃣1️⃣ TimeSeriesSplit — Leakage-Free Cross-Validation

> Standard K-Fold shuffles data — **never use it for time series!**  
> `TimeSeriesSplit` uses expanding windows: train always precedes test.
>
> ```
> Split 1: [TRAIN──────────][TEST]
> Split 2: [TRAIN──────────────][TEST]
> Split 3: [TRAIN──────────────────][TEST]
> ```


In [ ]:
# ── TimeSeriesSplit CV ────────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5, gap=0)

# Compare models
models = {
    'Ridge'          : Ridge(alpha=1.0),
    'RandomForest'   : RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100,
                                                    learning_rate=0.05,
                                                    max_depth=3, random_state=42),
}

X_arr = X.values
y_arr = y.values

print('TimeSeriesSplit CV Results (5-fold, expanding window):')
cv_rows = []

for name, model in models.items():
    fold_rmse = []
    fold_r2   = []
    for train_idx, test_idx in tscv.split(X_arr):
        X_tr, X_te = X_arr[train_idx], X_arr[test_idx]
        y_tr, y_te = y_arr[train_idx], y_arr[test_idx]

        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)

        model.fit(X_tr_sc, y_tr)
        y_pred = model.predict(X_te_sc)

        fold_rmse.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        fold_r2.append(r2_score(y_te, y_pred))

    cv_rows.append({
        'Model'     : name,
        'RMSE Mean' : round(np.mean(fold_rmse), 4),
        'RMSE Std'  : round(np.std(fold_rmse), 4),
        'R² Mean'   : round(np.mean(fold_r2), 4),
        'R² Std'    : round(np.std(fold_r2), 4),
    })
    print(f'  {name:20s}: RMSE={np.mean(fold_rmse):.4f} ± {np.std(fold_rmse):.4f} '
          f'| R²={np.mean(fold_r2):.4f} ± {np.std(fold_r2):.4f}')

cv_df = pd.DataFrame(cv_rows).sort_values('R² Mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(cv_df)); w = 0.5
axes[0].bar(cv_df['Model'], cv_df['RMSE Mean'],
            yerr=cv_df['RMSE Std'], color=COLORS['secondary'],
            alpha=0.85, edgecolor='white', capsize=6)
axes[0].set_ylabel('RMSE', fontsize=11)
axes[0].set_title('TimeSeriesSplit CV — RMSE per Model', fontsize=12, fontweight='bold')

axes[1].bar(cv_df['Model'], cv_df['R² Mean'],
            yerr=cv_df['R² Std'], color=COLORS['accent'],
            alpha=0.85, edgecolor='white', capsize=6)
axes[1].set_ylabel('R²', fontsize=11)
axes[1].set_title('TimeSeriesSplit CV — R² per Model', fontsize=12, fontweight='bold')

plt.suptitle('TimeSeriesSplit Cross-Validation Results',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Visualize splits ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_arr)):
    ax.barh(fold, len(train_idx), left=0,
            color=COLORS['primary'], alpha=0.5, height=0.6)
    ax.barh(fold, len(test_idx), left=len(train_idx),
            color=COLORS['secondary'], alpha=0.85, height=0.6)
    ax.text(len(train_idx) + len(test_idx) + 2, fold,
            f'Test: {len(test_idx)} days', va='center', fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Sample Index (days)', fontsize=11)
ax.set_title('TimeSeriesSplit — Expanding Train Windows',
             fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['primary'],   alpha=0.5, label='Train'),
    Patch(color=COLORS['secondary'], alpha=0.85, label='Test'),
], fontsize=10)
plt.tight_layout(); plt.show()

---
## ✅ 12. Summary & Golden Rules

| Feature Type | Function | Created With |
|-------------|----------|-------------|
| **Lag features** | Past values as predictors | `shift(lag)` |
| **Rolling mean** | Local trend / smoothed level | `shift(h).rolling(w).mean()` |
| **Rolling std** | Local volatility | `shift(h).rolling(w).std()` |
| **Expanding mean** | Long-run historical average | `shift(h).expanding().mean()` |
| **Calendar (cyclical)** | Periodic patterns | `sin/cos(2π×t/period)` |
| **Differencing** | Remove trend | `.diff(1)` |
| **Seasonal diff** | Remove seasonality | `.diff(period)` |

### 🔑 Golden Rules

1. **Never shuffle time series data** — always respect temporal order
2. **Only use lags ≥ forecast horizon** — shorter lags cause lookahead bias
3. **Always `shift(h)` before rolling/expanding** — avoids using current value
4. **Use sin/cos for cyclical features** — not raw integers (month, hour, weekday)
5. **Fit scaler only on train fold** — never on full dataset before split
6. **Use TimeSeriesSplit** for CV — never standard K-Fold
7. **Test stationarity before modeling** — ADF + KPSS together
8. **Difference first, then feature-engineer** — or include diff as a feature
9. **Include multiple lag windows** — short (1–7), medium (14–28), long (90+)
10. **More features ≠ better** — use feature importance to prune after initial run

---

## 🔗 Next Steps

- ➡️ `09_Time_Series_Machine_Learning/preprocessing.md` — Core preprocessing theory
- ➡️ `09_Time_Series_Machine_Learning/forecasting_models.ipynb` — ARIMA, Prophet, ML models
- ➡️ `06_Feature_Selection/embedded_methods.ipynb` — Select best time series features
- ➡️ `07_Hyperparameter_Tuning/randomsearchcv.ipynb` — Tune GBM for time series
